In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, hamming_loss, classification_report

In [4]:
df = pd.read_csv("datasets/studentsupport.csv")
display(df)
df.head()

In [5]:
print('Dataset shape:', df.shape)
print('Column information:')
print(df.info())

print('Support label counts:')
print(df[['MathSupport', 'EnglishSupport', 'AttendanceSupport']].sum())

Dataset shape: (20, 9)
Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Student            20 non-null     object
 1   Attendance         20 non-null     int64 
 2   MathScore          20 non-null     int64 
 3   EnglishScore       20 non-null     int64 
 4   QuizScore          20 non-null     int64 
 5   MissedClasses      20 non-null     int64 
 6   MathSupport        20 non-null     int64 
 7   EnglishSupport     20 non-null     int64 
 8   AttendanceSupport  20 non-null     int64 
dtypes: int64(8), object(1)
memory usage: 1.5+ KB
None
Support label counts:
MathSupport          7
EnglishSupport       9
AttendanceSupport    3
dtype: int64


In [6]:
feature_columns = [
    'Attendance', 'MathScore', 'EnglishScore', 'QuizScore', 'MissedClasses'
]

target_columns = [
    'MathSupport', 'EnglishSupport', 'AttendanceSupport'
]

X = df[feature_columns]
y = df[target_columns]

print('X (features):')
display(X.head())

print('y (multiple support labels):')
display(y.head())

X (features):


,Attendance,MathScore,EnglishScore,QuizScore,MissedClasses
0,72,55,58,61,1
1,91,86,84,88,0
2,68,48,52,55,4
3,94,92,89,91,0
4,75,60,57,63,2


y (multiple support labels):


,MathSupport,EnglishSupport,AttendanceSupport
0,0,1,0
1,0,0,0
2,1,1,0
3,0,0,0
4,0,1,0


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

Training samples: 16
Testing samples: 4


In [8]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', OneVsRestClassifier(
        LogisticRegression(max_iter=1000)
    ))
])

model.fit(X_train, y_train)
print('Model training completed successfully.')

Model training completed successfully.


In [9]:
y_pred = model.predict(X_test)

results = X_test.copy()
results['Actual_MathSupport'] = y_test['MathSupport'].values
results['Predicted_MathSupport'] = y_pred[:, 0]
results['Actual_EnglishSupport'] = y_test['EnglishSupport'].values
results['Predicted_EnglishSupport'] = y_pred[:, 1]
results['Actual_AttendanceSupport'] = y_test['AttendanceSupport'].values
results['Predicted_AttendanceSupport'] = y_pred[:, 2]

display(results)

print('Exact-match accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print('Hamming loss:', round(hamming_loss(y_test, y_pred), 4))
print('\nClassification report:')
print(classification_report(
    y_test, y_pred,
    target_names=target_columns,
    zero_division=0
))

,Attendance,MathScore,EnglishScore,QuizScore,MissedClasses,Actual_MathSupport,Predicted_MathSupport,Actual_EnglishSupport,Predicted_EnglishSupport,Actual_AttendanceSupport,Predicted_AttendanceSupport
0,72,55,58,61,1,0,0,1,1,0,0
17,89,82,79,84,1,0,0,0,0,0,0
15,77,68,66,70,2,0,0,0,0,0,0
1,91,86,84,88,0,0,0,0,0,0,0


Exact-match accuracy: 100.0 %
Hamming loss: 0.0

Classification report:
                   precision    recall  f1-score   support

      MathSupport       0.00      0.00      0.00         0
   EnglishSupport       1.00      1.00      1.00         1
AttendanceSupport       0.00      0.00      0.00         0

        micro avg       1.00      1.00      1.00         1
        macro avg       0.33      0.33      0.33         1
     weighted avg       1.00      1.00      1.00         1
      samples avg       0.25      0.25      0.25         1



In [10]:
def display_support_prediction(student_name, prediction):
    print(f'Student Support Prediction: {student_name}')
    print(f'Math Support:       {"YES" if prediction[0] == 1 else "NO"}')
    print(f'English Support:    {"YES" if prediction[1] == 1 else "NO"}')
    print(f'Attendance Support: {"YES" if prediction[2] == 1 else "NO"}')


for student_name, prediction in zip(df.loc[X_test.index, 'Student'], y_pred):
    display_support_prediction(student_name, prediction)

Student Support Prediction: Aarav
Math Support:       NO
English Support:    YES
Attendance Support: NO
Student Support Prediction: Sanjay
Math Support:       NO
English Support:    NO
Attendance Support: NO
Student Support Prediction: Rita
Math Support:       NO
English Support:    NO
Attendance Support: NO
Student Support Prediction: Sita
Math Support:       NO
English Support:    NO
Attendance Support: NO


In [11]:
new_student = pd.DataFrame({
    'Attendance': [66],
    'MathScore': [46],
    'EnglishScore': [49],
    'QuizScore': [52],
    'MissedClasses': [5]
})

new_prediction = model.predict(new_student)[0]
display_support_prediction('New Student', new_prediction)

Student Support Prediction: New Student
Math Support:       YES
English Support:    YES
Attendance Support: YES


## 9. Conclusion

The model predicts three support requirements independently using `OneVsRestClassifier`. Because this is a multilabel problem, a student may need **one, two, or all three types of support simultaneously**.

**Note:** The dataset contains only 20 students, so the test score can change depending on the train/test split. The result should therefore be treated as a demonstration of the assignment rather than a highly reliable real-world model.